# BÁO CÁO THỰC NGHIỆM LAB 01: TỪ XỬ LÝ VĂN BẢN ĐẾN TÌM KIẾM

Môn học: Xử lý ngôn ngữ tự nhiên và ứng dụng  
Sinh viên: Nguyễn Hồng Quân - 23001921  

---

Trong notebook này, em thực hiện các phần thực nghiệm từ **Phần 7 (Part D)** đến hết **Phần 13 (Part J)** theo đúng hướng dẫn của đề bài, sử dụng tập dữ liệu C4 gồm 30.000 văn bản.


In [30]:
import json
import os
import re
import time
import math
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import tiktoken
import nltk
from nltk.corpus import stopwords

# Tải danh sách stop words tiếng Anh nếu chưa có
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

print("Đã tải xong các thư viện cần thiết!")


Đã tải xong các thư viện cần thiết!


## 7. Part D — Thí nghiệm 1: Khám phá ma trận biểu diễn thưa (Sparse Representation)

### 7.1 & 7.2 Đọc dữ liệu và xây dựng Pipeline
Em nạp 30.000 văn bản từ file dữ liệu `c4-train.00000-of-01024-30K.json` và dùng `TfidfVectorizer` của thư viện scikit-learn để chuyển các văn bản thành ma trận TF-IDF.


In [31]:
# Tìm file dữ liệu c4-train trong thư mục hiện tại hoặc lân cận
data_file = "c4-train.00000-of-01024-30K.json"
if not os.path.exists(data_file):
    if os.path.exists("lab_1/c4-train.00000-of-01024-30K.json"):
        data_file = "lab_1/c4-train.00000-of-01024-30K.json"

docs = []
with open(data_file, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            docs.append(json.loads(line)["text"])

N = len(docs)

# Dùng TfidfVectorizer để tạo ma trận
tfidf_vec = TfidfVectorizer()
X_tfidf = tfidf_vec.fit_transform(docs)

V = len(tfidf_vec.vocabulary_)
nnz = X_tfidf.nnz
total_cells = N * V
sparsity = 1.0 - (nnz / total_cells)
feature_names = np.array(tfidf_vec.get_feature_names_out())

print(f"Số lượng văn bản (N) = {N}")
print(f"Kích thước từ vựng (V) = {V}")
print(f"Kích thước ma trận = {X_tfidf.shape}")
print(f"Số phần tử khác 0 (nnz) = {nnz:,}")
print(f"Tổng số ô trong ma trận (N x V) = {total_cells:,}")
print(f"Độ thưa của ma trận (Sparsity S) = {sparsity * 100:.4f}%")


Số lượng văn bản (N) = 30000
Kích thước từ vựng (V) = 193540
Kích thước ma trận = (30000, 193540)
Số phần tử khác 0 (nnz) = 4,985,822
Tổng số ô trong ma trận (N x V) = 5,806,200,000
Độ thưa của ma trận (Sparsity S) = 99.9141%


### 7.4 Trả lời câu hỏi về Sparsity

**Câu hỏi:** Tại sao một document chỉ sử dụng một phần rất nhỏ vocabulary nhưng vector vẫn có chiều V?

**Trả lời:**  
1. Để có thể tính toán so sánh giữa hai văn bản bất kỳ (ví dụ như tính góc cosine similarity giữa văn bản A và văn bản B), hai vector của chúng bắt buộc phải nằm trong cùng một không gian tọa độ chung, tức là mỗi vị trí cột phải đại diện cho cùng một từ.
2. Một văn bản bình thường chỉ có tầm 100 đến 200 từ duy nhất, trong khi cả kho dữ liệu có tới gần 200.000 từ. Những từ nào không xuất hiện trong bài viết đó thì đành phải gán giá trị bằng 0.
3. Vì thế mà ma trận bị thưa tới hơn **99.91%** là số 0, và ta bắt buộc phải dùng cấu trúc ma trận thưa (`csr_matrix`) để không bị tràn bộ nhớ RAM.


In [32]:
# Tính Document Frequency cho từng từ
df = np.diff(X_tfidf.tocsc().indptr)

# 1. 20 từ xuất hiện phổ biến nhất
top_df_idx = np.argsort(df)[::-1][:20]
df_table = pd.DataFrame({
    "Hạng": range(1, 21),
    "Từ": feature_names[top_df_idx],
    "Số bài chứa từ (DF)": df[top_df_idx],
    "Điểm IDF": tfidf_vec.idf_[top_df_idx].round(4)
})

# 2. 20 từ có IDF cao nhất
top_idf_idx = np.argsort(tfidf_vec.idf_)[::-1][:20]
idf_table = pd.DataFrame({
    "Hạng": range(1, 21),
    "Từ": feature_names[top_idf_idx],
    "DF": df[top_idf_idx],
    "Điểm IDF": tfidf_vec.idf_[top_idf_idx].round(4)
})

# 3. 20 từ có điểm TF-IDF cao nhất trong bài viết đầu tiên (Doc 0 - lớp dạy nấu BBQ)
doc0_weights = X_tfidf[0].toarray().flatten()
top_doc0_idx = np.argsort(doc0_weights)[::-1][:20]
doc0_table = pd.DataFrame({
    "Hạng": range(1, 21),
    "Từ khóa": feature_names[top_doc0_idx],
    "Điểm TF-IDF": doc0_weights[top_doc0_idx].round(4)
})

print("--- 20 TỪ CÓ TẦN SUẤT XUẤT HIỆN CAO NHẤT (TOP DF) ---")
print(df_table.head(10).to_string(index=False))

print("\n--- 20 TỪ CÓ ĐIỂM IDF CAO NHẤT ---")
print(idf_table.head(10).to_string(index=False))

print("\n--- 20 TỪ CÓ TF-IDF CAO NHẤT TRONG BÀI VIẾT 0 (NẤU BBQ) ---")
print(doc0_table.head(10).to_string(index=False))


--- 20 TỪ CÓ TẦN SUẤT XUẤT HIỆN CAO NHẤT (TOP DF) ---
 Hạng   Từ  Số bài chứa từ (DF)  Điểm IDF
    1  the                27893    1.0728
    2  and                27423    1.0898
    3   to                26689    1.1169
    4   of                26031    1.1419
    5   in                25224    1.1734
    6  for                23651    1.2378
    7   is                22739    1.2771
    8 with                21405    1.3376
    9   on                20262    1.3924
   10 that                18370    1.4905

--- 20 TỪ CÓ ĐIỂM IDF CAO NHẤT ---
 Hạng         Từ  DF  Điểm IDF
    1      00000   1   10.6158
    2      00003   1   10.6158
    3     000040   1   10.6158
    4      00005   1   10.6158
    5 0000856166   1   10.6158
    6    0001042   1   10.6158
    7     000116   1   10.6158
    8      00012   1   10.6158
    9      00015   1   10.6158
   10      00016   1   10.6158

--- 20 TỪ CÓ TF-IDF CAO NHẤT TRONG BÀI VIẾT 0 (NẤU BBQ) ---
 Hạng  Từ khóa  Điểm TF-IDF
    1      bbq    

### 7.5 So sánh 3 danh sách và trả lời câu hỏi

**So sánh nhanh:**
- Danh sách DF cao toàn là các từ dừng quen thuộc (*the, and, to, of, in*), câu nào cũng có nên điểm IDF rất thấp (chỉ quanh mức 1.0 đến 1.5).
- Danh sách IDF cao nhất lại là các từ chỉ xuất hiện đúng 1 lần trong cả 30.000 bài viết (DF = 1), gồm mã số lạ hoặc lỗi gõ phím.
- Danh sách TF-IDF trong bài viết 0 chọn lọc rất chuẩn các từ quan trọng nói về lớp dạy nấu món nướng BBQ: *bbq, class, meat, smoker, apron*.

**Câu hỏi 1: Một term xuất hiện rất nhiều trong corpus có nhất thiết có TF-IDF cao không?**  
- **Không.** Vì từ xuất hiện nhiều thì DF rất lớn, kéo theo IDF rất bé (xấp xỉ 0). Khi nhân TF với một số gần bằng 0 thì điểm TF-IDF sẽ bị kéo xuống rất thấp.

**Câu hỏi 2: Một term có IDF cao có nhất thiết có TF-IDF cao trong mọi document không?**  
- **Không.** Vì nếu từ đó không xuất hiện trong bài viết đang xét thì số lần xuất hiện bằng 0 (TF = 0), nhân với IDF cao cỡ nào thì kết quả TF-IDF trong bài đó vẫn bằng 0.


## 8. Part E — Tự cài đặt các hàm cốt lõi (Core Implementation)

Em đã tự viết các hàm tính toán cơ bản từ đầu trong file implementation.py:
- build_vocabulary()
- compute_counts()
- compute_tf()
- compute_idf()
- compute_tfidf()
- cosine_similarity()

Đoạn code dưới đây import file đó vào để chạy bộ Unit Test và đối chiếu với thư viện scikit-learn.


In [33]:
# Import các hàm tự viết từ file implementation.py
from implementation import run_unit_tests, compare_with_scikit_learn

# Chạy unit tests trên 3 câu mẫu (cat eats fish, dog eats fish, cat likes fish)
run_unit_tests()

# Đối chiếu kết quả với thư viện scikit-learn
compare_with_scikit_learn()


RUNNING UNIT TESTS ON TOY CORPUS (Part E)
[PASS] build_vocabulary(): ['cat', 'dog', 'eats', 'fish', 'likes']
[PASS] compute_counts(): shape (3, 5)
[PASS] compute_tf(): doc 0 sum = 1.0
[PASS] compute_idf() standard: idf(cat) = 0.4054651081081644 , idf(fish) = 0.0
[PASS] compute_tfidf() standard: tfidf(cat, D1) = 0.13515503603605478 , tfidf(fish, D1) = 0.0
[PASS] cosine_similarity([1,1,1], [1,1,0]) = 0.8164965809277259 ≈ 2/sqrt(6)
[PASS] Corpus TF-IDF similarity D1-D2: 0.2448, D1-D3: 0.2448

ALL UNIT TESTS PASSED SUCCESSFULLY!

COMPARISON: STUDENT IMPLEMENTATION vs SCIKIT-LEARN REFERENCE
Vocabulary match: True
Scikit-learn IDF values:
  [1.28768207 1.69314718 1.28768207 1.         1.69314718]
Student smoothed IDF values:
  [1.28768207 1.69314718 1.28768207 1.         1.69314718]
[MATCH] IDF values match exactly with scikit-learn!

Scikit-learn TF-IDF matrix (D1):
  [0.61980538 0.         0.61980538 0.48133417 0.        ]
Student TF-IDF matrix (D1 with sklearn mode):
  [0.61980538 0.     

### 8.5 Nhận xét sự khác nhau giữa công thức bài học và thư viện

Khi so sánh code tự viết với scikit-learn, em nhận thấy 2 điểm khác biệt chính:
1. **Làm mịn IDF (Smoothing):** Công thức trong slide dùng `ln(N / df)`, nên từ xuất hiện ở mọi câu như "fish" sẽ có IDF = 0. Trong khi scikit-learn cộng thêm 1 vào cả tử số, mẫu số và kết quả cuối để không có từ nào bị bằng 0.
2. **Chuẩn hóa vector (L2 normalization):** Scikit-learn luôn chia vector cho độ dài của nó để đưa về độ dài bằng 1, giúp cho việc tính cosine similarity sau này chỉ cần nhân ma trận là xong.


## 9. Part F — Thí nghiệm 2: So sánh các cách tiền xử lý (Preprocessing Ablation)

Em thử nghiệm 3 cách tiền xử lý khác nhau:
- **Pipeline A (Tối giản):** Chỉ chuyển chữ thường và tách từ theo khoảng trắng thông thường.
- **Pipeline B (Chuẩn hóa):** Chuyển chữ thường, bỏ dấu câu và loại bỏ các từ dừng (stopwords).
- **Pipeline C (Nâng cao):** Dùng thuật toán tách từ con BPE (Byte-Pair Encoding) của mô hình GPT-2 thông qua thư viện `tiktoken`.


In [34]:
# 1. Pipeline A: Tối giản
vec_A = CountVectorizer(lowercase=True, token_pattern=r'\S+')
X_A = vec_A.fit_transform(docs)
vocab_A = len(vec_A.vocabulary_)
avg_tok_A = X_A.sum() / N
sp_A = 1.0 - (X_A.nnz / (N * vocab_A))

# 2. Pipeline B: Chuẩn hóa
vec_B = CountVectorizer(lowercase=True, stop_words='english', token_pattern=r'(?u)\b[a-zA-Z]{2,}\b')
X_B = vec_B.fit_transform(docs)
vocab_B = len(vec_B.vocabulary_)
avg_tok_B = X_B.sum() / N
sp_B = 1.0 - (X_B.nnz / (N * vocab_B))

# 3. Pipeline C: Subwords BPE (GPT-2 tiktoken)
enc = tiktoken.get_encoding("gpt2")
vocab_C = enc.n_vocab
sample = docs[:500]
avg_tok_C = np.mean([len(enc.encode(d, disallowed_special=())) for d in sample])
avg_uniq_C = np.mean([len(set(enc.encode(d, disallowed_special=()))) for d in sample])
sp_C = 1.0 - (avg_uniq_C / vocab_C)

bang_so_sanh = pd.DataFrame({
    "Phương pháp": ["Pipeline A (Tối giản)", "Pipeline B (Chuẩn hóa)", "Pipeline C (Subwords BPE)"],
    "Kích thước từ vựng": [f"{vocab_A:,}", f"{vocab_B:,}", f"{vocab_C:,}"],
    "Số từ trung bình / bài": [f"{avg_tok_A:.1f}", f"{avg_tok_B:.1f}", f"{avg_tok_C:.1f}"],
    "Độ thưa ma trận": [f"{sp_A*100:.4f}%", f"{sp_B*100:.4f}%", f"{sp_C*100:.4f}%"],
    "Tỷ lệ từ lạ (OOV)": ["0.0%", "0.0%", "0.0%"]
})

print(bang_so_sanh.to_string(index=False))


              Phương pháp Kích thước từ vựng Số từ trung bình / bài Độ thưa ma trận Tỷ lệ từ lạ (OOV)
    Pipeline A (Tối giản)            473,388                  361.1        99.9611%              0.0%
   Pipeline B (Chuẩn hóa)            167,083                  183.5        99.9296%              0.0%
Pipeline C (Subwords BPE)             50,257                  500.8        99.5602%              0.0%


### 9.5 Trả lời 6 câu hỏi phân tích tiền xử lý

1. **Chuyển chữ thường (Lowercasing) thay đổi từ vựng như thế nào?**  
   Nó giúp gom các từ viết hoa (ở đầu câu hoặc viết hoa cả từ) về cùng dạng chữ thường, làm giảm kích thước từ vựng và tránh việc cùng một từ nhưng bị đếm thành 2 vector khác nhau.

2. **Xóa stop words có luôn tốt hơn không?**  
   Không hẳn. Khi tìm kiếm chung chung thì bỏ stopwords giúp lọc nhiễu rất tốt. Nhưng với các cụm từ đặc biệt mà ý nghĩa nằm ở stopwords (như *"to be or not to be"* hoặc *"Vitamin A"*), nếu xóa đi thì câu sẽ bị mất nghĩa.

3. **Bỏ dấu câu có thể làm mất thông tin gì?**  
   Làm mất các thông tin định dạng như địa chỉ web (`klyq.com`), giá tiền (`$35`), số phiên bản (`10.7`), hoặc ngôn ngữ lập trình (`C++`).

4. **Pipeline nào tạo ra ma trận thưa nhất?**  
   Pipeline A là thưa nhất (99.96% là số 0), vì từ vựng bị phình to tới hơn 473.000 từ do dính các ký tự rác.

5. **Pipeline nào cho tìm kiếm tốt nhất?**  
   Pipeline B tốt nhất cho bài toán tìm kiếm từ khóa, vì nó loại bỏ được các từ rác và từ dừng, chỉ giữ lại những từ chứa nội dung thực sự.

6. **Tìm kiếm tốt hơn có đồng nghĩa với từ vựng nhỏ hơn không?**  
   Không đồng nghĩa. Nếu cố làm từ vựng nhỏ quá bằng cách cắt từ quá đà thì nhiều từ khác nghĩa nhau sẽ bị gộp nhầm thành một, làm giảm chất lượng tìm kiếm.


## 10. Part G — Ứng dụng: Xây dựng công cụ tìm kiếm văn bản

Em xây dựng class `TfidfSearchEngine` đơn giản: khi người dùng nhập câu truy vấn (query), hệ thống sẽ biến câu đó thành vector TF-IDF, tính cosine similarity với toàn bộ 30.000 bài viết và xếp hạng lấy ra Top 5 bài viết có điểm cao nhất.


In [35]:
class TfidfSearchEngine:
    def __init__(self):
        self.vectorizer = TfidfVectorizer(
            lowercase=True,
            stop_words='english',
            token_pattern=r'(?u)\b[a-zA-Z]{2,}\b'
        )
        self.doc_matrix = None
        self.documents = []

    def index(self, documents):
        self.documents = documents
        self.doc_matrix = self.vectorizer.fit_transform(documents)

    def search(self, query: str, top_k: int = 5):
        q_vec = self.vectorizer.transform([query])
        # Tích vô hướng giữa ma trận tài liệu và vector query
        scores = self.doc_matrix.dot(q_vec.T).toarray().flatten()
        top_indices = np.argsort(scores)[::-1][:top_k]
        
        results = []
        for rank, idx in enumerate(top_indices, 1):
            results.append({
                "Hạng": rank,
                "Mã bài (Doc ID)": int(idx),
                "Điểm tương đồng": round(float(scores[idx]), 4),
                "Trích dẫn ngắn": self.documents[idx].replace("\n", " ")[:110] + "..."
            })
        return pd.DataFrame(results)

engine = TfidfSearchEngine()
engine.index(docs)

queries_demo = [
    "medical image classification",
    "transformer language model",
    "deep learning healthcare",
    "natural language processing"
]

for q in queries_demo:
    print(f"\n>>> Kết quả tìm kiếm cho: '{q}'")
    res_df = engine.search(q, top_k=5)
    print(res_df.to_string(index=False))



>>> Kết quả tìm kiếm cho: 'medical image classification'
 Hạng  Mã bài (Doc ID)  Điểm tương đồng                                                                                                    Trích dẫn ngắn
    1            18971           0.4274 The new RTS Environmental Classification system (RTS GLT) is designed for parties who are commissioning constr...
    2             8527           0.3537 History of maize classification. How races used in classification. Geographical distribution. Existing races o...
    3            17794           0.2660 Filters the output of 'wp_calculate_image_sizes()'. A source size value for use in a 'sizes' attribute. Reques...
    4            19908           0.2614 Download League Of Legends Wallpapers in high-quality for your desktop and smart-phone in wide-screen and HD r...
    5            15682           0.2505 - Group Image: Provided functionality of group's image, user can now view and change the group's image also. -...

>>> Kết quả tìm k

## 11. Part H — Đánh giá định lượng (Evaluation)

Em tạo một tập gồm 8 queries và đo lường 3 chỉ số chính theo yêu cầu:
- **Precision@5:** Tỷ lệ tài liệu đúng trong top 5 kết quả tìm được.
- **Recall@5:** Tỷ lệ tài liệu đúng tìm được so với tổng số tài liệu đúng có trong kho.
- **MRR (Mean Reciprocal Rank):** Điểm thứ hạng của tài liệu đúng đầu tiên xuất hiện.


In [36]:
# Đọc bảng kết quả đã được tính toán và lưu trong file results.csv
results_df = pd.read_csv("results.csv")
print("=== BẢNG KẾT QUẢ ĐÁNH GIÁ 8 TRUY VẤN ===")
print(results_df[["query_id", "query_text", "precision@5", "recall@5", "mrr", "total_relevant_in_corpus"]].to_string(index=False))

print(f"\nĐiểm Precision@5 trung bình : {results_df['precision@5'].mean():.4f}")
print(f"Điểm Recall@5 trung bình    : {results_df['recall@5'].mean():.4f}")
print(f"Điểm MRR trung bình         : {results_df['mrr'].mean():.4f}")


=== BẢNG KẾT QUẢ ĐÁNH GIÁ 8 TRUY VẤN ===
query_id                    query_text  precision@5  recall@5  mrr  total_relevant_in_corpus
      Q1  medical image classification          0.0    0.0000  0.0                         4
      Q2    transformer language model          0.0    0.0000  0.0                         0
      Q3      deep learning healthcare          0.0    0.0000  0.0                        10
      Q4   natural language processing          0.0    0.0000  0.0                        17
      Q5       bbq cooking meat smoker          0.2    1.0000  0.5                         1
      Q6  macbook restore disk utility          0.2    0.1667  0.5                         6
      Q7       heart attack prevention          0.0    0.0000  0.0                         6
      Q8 myocardial infarction therapy          0.4    0.3333  0.5                         6

Điểm Precision@5 trung bình : 0.1000
Điểm Recall@5 trung bình    : 0.1875
Điểm MRR trung bình         : 0.1875


## 12. Part I — Phân tích lỗi (Error Analysis)

### 1. Hai truy vấn có kết quả tốt:
- **Query Q5 (`"bbq cooking meat smoker"`):** Tìm ra ngay bài viết Doc 0 (lớp học nấu BBQ), vì các từ `bbq`, `smoker` rất đặc thù và hiếm gặp trong toàn bộ kho dữ liệu.
- **Query Q8 (`"myocardial infarction therapy"`):** Tìm được đúng các bài báo y học điều trị nhồi máu cơ tim, vì từ `myocardial` và `infarction` có điểm IDF rất cao.

### 2. Hai truy vấn gặp lỗi (Failure cases):
- **Query Q2 (`"transformer language model"`):**  
  Bài viết đứng đầu là Doc 27936 nói về sửa mạch điện máy biến áp của mô hình xe hơi.  
  *Nguyên nhân:* Bị lỗi **từ đa nghĩa (Polysemy)**. Từ "transformer" vừa có nghĩa là mô hình AI vừa có nghĩa là cục biến áp.
- **Query Q7 (`"heart attack prevention"`):**  
  Hệ thống không tìm ra tài liệu y khoa mong muốn mà toàn ra bài viết về tấn công mạng (`cyber attack`).  
  *Nguyên nhân:* Thiếu từ khóa trùng khớp (**Lexical mismatch**). Trong khi tài liệu y khoa dùng thuật ngữ chuyên môn là "myocardial infarction", người dùng lại tìm từ thông dụng "heart attack", khiến TF-IDF tính ra điểm trùng nhau bằng 0.


## 13. Part J — Hướng phát triển mô hình tiếp theo

Qua các lỗi trên, em thấy rõ hạn chế lớn nhất của TF-IDF là **chỉ nhìn vào mặt chữ chứ không hiểu nghĩa của từ**:
- "heart attack" và "myocardial infarction" cùng một nghĩa nhưng bị coi là hai vector vuông góc với nhau (độ tương đồng = 0).
- "transformer" trong mô hình AI và "transformer" máy biến áp bị coi là giống hệt nhau.

**Ý tưởng cho các bài học tiếp theo:**  
Ta cần các phương pháp biểu diễn từ dưới dạng vector liên tục (Dense Embeddings):
1. **Word Embeddings (như Word2Vec):** Các từ hay xuất hiện trong cùng ngữ cảnh (như *heart attack* và *cardiac*) sẽ có vector nằm gần nhau.
2. **Contextual Embeddings (như BERT/Transformer):** Ý nghĩa vector của từ sẽ thay đổi tùy thuộc vào câu văn xung quanh, giúp phân biệt rõ máy biến áp với mô hình ngôn ngữ AI.
